In [33]:
import pandas as pd
import numpy as np
from collections import Counter
import pickle

from Evaluation import evaluate_comprehensive, evaluate_light

In [25]:
def add_length_bucket(
    df,
    length_buckets,
    case_id_col="case:concept:name",
    time_col="time:timestamp",
    bucket_col="length_bucket",
    pos_col=None
):
    """
    Assign a length bucket to each event row based on the trace length.

    Parameters
    ----------
    df : pd.DataFrame
        Event log dataframe, one row per event.
    length_buckets : dict
        Example:
        {
            "short":  (2, 5),
            "medium": (5, 7),
            "long":   (7, None)
        }
    case_id_col : str
        Case id column name.
    time_col : str
        Timestamp column name. Used for stable ordering if pos_col is not given.
    bucket_col : str
        Name of output bucket column.
    pos_col : str or None
        Optional event position column. If provided, sort by [case_id_col, pos_col].
        Otherwise sort by [case_id_col, time_col].

    Returns
    -------
    pd.DataFrame
        Copy of input with new bucket column.
    """
    out = df.copy()

    # stable ordering before group size / later comparisons
    if pos_col is not None and pos_col in out.columns:
        out = out.sort_values([case_id_col, pos_col]).reset_index(drop=True)
    else:
        out = out.sort_values([case_id_col, time_col]).reset_index(drop=True)

    # trace length per case
    case_lengths = out.groupby(case_id_col).size()

    # map case -> bucket
    case_to_bucket = {}
    for cid, length in case_lengths.items():
        assigned = None
        for bucket_name, (lo, hi) in length_buckets.items():
            if hi is None:
                if length >= lo:
                    assigned = bucket_name
                    break
            else:
                if lo <= length < hi:
                    assigned = bucket_name
                    break
        case_to_bucket[cid] = assigned

    out[bucket_col] = out[case_id_col].map(case_to_bucket)

    return out

In [26]:
def filter_by_length_bucket(df, bucket, case_id_col, bucket_col="length_bucket"):
    cases = df[df[bucket_col] == bucket][case_id_col].unique()
    return df[df[case_id_col].isin(cases)]

In [38]:
case_index = 'case:concept:name'
time_col = 'time:timestamp'
core_event = "concept:name"
delta_col='delta_time'

In [37]:
#datanames = ["helpdesk", "sepsis", "BPI13I", "BPI13C"]
#dataname = "BPI12W"
#datanames = ["BPI13I"]
#dataname = "BPI17"
datanames = ["sepsis"]
#dataname = "BPI12W"
#dataname = "BPI12"
#datanames = ["BPI17"]
#datanames = ["helpdesk", "sepsis", "BPI13I", "BPI20", "BPI13C", "BPI17"]

In [27]:
for dataname in datanames:

    model_files = {
    "GGATN": f"../output/gen_traces/{dataname}_gen_gat.csv",
    "GGATN_j": f"../output/gen_traces/{dataname}_gen_gat_joint.csv",
    "GGATN_s5": f"../output/gen_traces/{dataname}_gen_gat_joint_stage.csv",
    "GGATN_s10": f"../output/gen_traces/{dataname}_gen_gat_joint_stage10.csv",
}

    # --------------------------------------------------
    # length buckets
    # --------------------------------------------------
    if dataname == "helpdesk":
        length_buckets = {
            "short":  (2, 5),   # 271 traces
            "medium": (5, 7),   # 154 traces
            "long":   (7, None) # 32 traces (7–11 merged)
        }
    elif dataname == "sepsis":
        length_buckets = {
        "short":  (3, 8),
        "medium": (8, 18),
        "long":   (18, None)
        }
    elif dataname == "BPI13I":
        length_buckets = {
        "short":  (1, 6),    # 1–5
        "medium": (6, 16),   # 6–15
        "long":   (16, None) # 16+
        }
    elif dataname == "BPI20":
        length_buckets = {
        "short":  (1, 8),    # 1–7
        "medium": (8, 12),   # 8–11
        "long":   (12, None) # 12+
        }
    elif dataname == "BPI12W":
        length_buckets = {
        "short":  (2, 8),     # 2–7
        "medium": (8, 20),    # 8–19
        "long":   (20, 46),   # 20–45
        "xlong":  (46, None)  # 46+
    }
    elif dataname == "BPI12":
        length_buckets = {
            "short":  (3, 8),     # 3–7
            "medium": (8, 24),    # 8–23
            "long":   (24, 61),   # 24–60
            "xlong":  (61, None)  # 61+
        }
    elif dataname == "BPI17":
        length_buckets = {
            "short":  (10, 21),    # 10–20
            "medium": (21, 41),    # 21–40
            "long":   (41, 61),    # 41–60
            "xlong":  (61, None)   # 61+
        }
    elif dataname == "BPI13C":
        length_buckets = {
        "short":  (1, 4),    # 1–3
        "medium": (4, 8),    # 4–7
        "long":   (8, None)  # 8+
    }

    # --------------------------------------------------
    # attribute setup
    # --------------------------------------------------

    if dataname == "helpdesk":
        cat_cols_event = ['org:resource']
        num_cols_event = []
        cat_cols_seq = ['case:variant']
        num_cols_seq = []
    elif dataname == "BPI12" or dataname == "BPI12W":
        num_cols_event = []
        cat_cols_seq = []
        num_cols_seq = ['case:AMOUNT_REQ']
    elif dataname == "BPI13I" or dataname == "BPI13C":
        cat_cols_event = ['org:group', "resource country", "org:resource", "organization involved", "org:role"]
        num_cols_event = []
        cat_cols_seq = ["organization country", "impact", "product"]
        num_cols_seq = []
    elif dataname == "sepsis":
        cat_cols_event = ['org:group']
        num_cols_event = ['Leucocytes', 'CRP', 'LacticAcid']
        cat_cols_seq = ['InfectionSuspected', 'DiagnosticBlood',     'DisfuncOrg',  'SIRSCritTachypnea', 'Hypotensie',       'SIRSCritHeartRate', 
                        'Infusion',           'DiagnosticArtAstrup', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther',
                        'SIRSCriteria2OrMore', 'DiagnosticXthorax',  'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 
                        'DiagnosticLacticAcid', 'Diagnose',          'Hypoxie',             'DiagnosticUrinarySediment', 'DiagnosticECG']
        num_cols_seq = [ 'Age']
    elif dataname == "BPI17":
        cat_cols_event = ['Action', 'org:resource', 'EventOrigin', 'Accepted', 'Selected', "OfferID"]
        num_cols_event = ['FirstWithdrawalAmount', 'NumberOfTerms', 'MonthlyCost',  'CreditScore', 'OfferedAmount']
        cat_cols_seq = [ 'case:LoanGoal', 'case:ApplicationType']
        num_cols_seq = ['case:RequestedAmount']
    elif dataname == "BPI20":
        cat_cols_event = ['org:role']
        num_cols_event = []
        cat_cols_seq = ["case:OrganizationalEntity", "case:Project"]
        num_cols_seq = ["case:RequestedAmount", "case:Permit RequestedBudget"]  


    gt_df = pd.read_csv("../output/data_processed/" + dataname + "_hold.csv")
    gt_df = add_length_bucket(
            gt_df,
            length_buckets=length_buckets,
            case_id_col=case_index,
            time_col=time_col,
            bucket_col="length_bucket",
            pos_col="pos"
        )

    all_eval = pd.read_csv("../output/metrics/" + dataname + "_eval.csv")
    
    bol_cols_event = []
    bol_cols_seq = []

    for model, file_path in model_files.items():
        gen_df = pd.read_csv(file_path)
        print(f"Evaluating {model}")

        gen_df = add_length_bucket(
            gen_df,
            length_buckets=length_buckets,
            case_id_col=case_index,
            time_col=time_col,
            bucket_col="length_bucket",
            pos_col="pos"
        )

        for bucket_name, _ in length_buckets.items():  
    
            gen_b = filter_by_length_bucket(gen_df, bucket_name, case_index)
            gt_b  = filter_by_length_bucket(gt_df,  bucket_name, case_index)
    
            # Skip empty buckets
            if gen_b.empty or gt_b.empty:
                continue
        
            
            eval_df =evaluate_comprehensive(gen_b, gt_b, case_index, core_event, time_col, "pos", 
                               [core_event] + cat_cols_event, num_cols_event, bol_cols_event,
                               cat_cols_seq, num_cols_seq, bol_cols_seq,
                               name=model, jsd_lambda=1.0
                              )
        
            # Add metadata columns
            eval_df["length_bucket"] = bucket_name
            eval_df["num_cases"] = gen_b[case_index].nunique()
            eval_df['data'] = dataname            
            eval_df = eval_df.reset_index().rename(columns={'index': 'model'})
            # ✅ THIS IS THE FIX
            all_eval = pd.concat([all_eval, eval_df], ignore_index=True)
            
    all_eval.to_csv("../output/metrics/" + dataname + "_eval_f.csv", index = False)

            

Evaluating GGATN
Evaluating GGATN_j
Evaluating GGATN_s5
Evaluating GGATN_s10


In [20]:
for dataname in datanames:
    df1 = pd.read_csv("../output/metrics/" + dataname + "_eval_f.csv")
    df2 = pd.read_csv("../output/metrics/" + dataname + "_eval_32.csv")
    res = pd.concat([df1, df2], axis=0, ignore_index=True)
    res.to_csv("../output/metrics/" + dataname + "_eval_final.csv", index = False)

In [39]:
for dataname in datanames:
    #df = pd.read_csv("../output/metrics/" + dataname + "_eval_final.csv")
    df = pd.read_csv("../output/metrics/" + dataname + "_eval.csv")
    filtered_df = df.loc[df['length_bucket'] == "all"]
    df_clean = filtered_df.loc[:, ~filtered_df.columns.str.endswith('_conditional')]
    df_clean.to_csv("../output/metrics/" + dataname + "_eval_paper.csv", index = False)

In [54]:
dataname = "BPI20"
gt_df = pd.read_csv("../output/data_processed/" + dataname + "_hold.csv")
gt_df.groupby(case_index).size().max()

np.int64(16)

In [34]:
def compute_dataset_stats(df, case_id_col, time_col, activity_col, length_buckets):
    df = df.sort_values([case_id_col, time_col])

    # basic counts
    num_cases = df[case_id_col].nunique()
    num_events = len(df)
    num_activities = df[activity_col].nunique()

    # trace lengths
    case_lengths = df.groupby(case_id_col).size()

    avg_len = case_lengths.mean()
    min_len = case_lengths.min()
    max_len = case_lengths.max()
    median_len = case_lengths.median()
    std_len = case_lengths.std()

    # duration (seconds)
    durations = df.groupby(case_id_col)[time_col].agg(
        lambda x: (x.max() - x.min()).total_seconds()
    )

    avg_duration = durations.mean()
    median_duration = durations.median()

    # inter-event time
    diffs = df.groupby(case_id_col)[time_col].diff().dt.total_seconds()
    avg_inter_event = diffs.mean()
    std_inter_event = diffs.std()

    # bucket counts
    bucket_counts = {}
    for bucket_name, (lo, hi) in length_buckets.items():
        if hi is None:
            count = (case_lengths >= lo).sum()
        else:
            count = ((case_lengths >= lo) & (case_lengths < hi)).sum()
        bucket_counts[bucket_name] = count

    return {
        "cases": num_cases,
        "events": num_events,
        "activities": num_activities,
        "avg_len": round(avg_len, 2),
        "min_len": int(min_len),
        "max_len": int(max_len),
        "median_len": round(median_len, 2),
        "std_len": round(std_len, 2),
        "avg_duration": round(avg_duration, 2),
        "median_duration": round(median_duration, 2),
        "avg_inter_event": round(avg_inter_event, 2),
        "std_inter_event": round(std_inter_event, 2),
        **bucket_counts
    }

In [35]:
stats_rows = []

for dataname in datanames:

    print(f"Processing {dataname}")

    df = pd.read_csv(f"../output/data_processed/{dataname}_full.csv")

    df[time_col] = pd.to_datetime(df[time_col])

    # --------------------------------------------------
    # length buckets
    # --------------------------------------------------
    if dataname == "helpdesk":
        length_buckets = {
            "short":  (2, 5),   # 271 traces
            "medium": (5, 7),   # 154 traces
            "long":   (7, None) # 32 traces (7–11 merged)
        }
    elif dataname == "sepsis":
        length_buckets = {
        "short":  (3, 8),
        "medium": (8, 18),
        "long":   (18, None)
        }
    elif dataname == "BPI13I":
        length_buckets = {
        "short":  (1, 6),    # 1–5
        "medium": (6, 16),   # 6–15
        "long":   (16, None) # 16+
        }
    elif dataname == "BPI20":
        length_buckets = {
        "short":  (1, 8),    # 1–7
        "medium": (8, 12),   # 8–11
        "long":   (12, None) # 12+
        }
    elif dataname == "BPI12W":
        length_buckets = {
        "short":  (2, 8),     # 2–7
        "medium": (8, 20),    # 8–19
        "long":   (20, 46),   # 20–45
        "xlong":  (46, None)  # 46+
    }
    elif dataname == "BPI12":
        length_buckets = {
            "short":  (3, 8),     # 3–7
            "medium": (8, 24),    # 8–23
            "long":   (24, 61),   # 24–60
            "xlong":  (61, None)  # 61+
        }
    elif dataname == "BPI17":
        length_buckets = {
            "short":  (10, 21),    # 10–20
            "medium": (21, 41),    # 21–40
            "long":   (41, 61),    # 41–60
            "xlong":  (61, None)   # 61+
        }
    elif dataname == "BPI13C":
        length_buckets = {
        "short":  (1, 4),    # 1–3
        "medium": (4, 8),    # 4–7
        "long":   (8, None)  # 8+
    }


    stats = compute_dataset_stats(
        df,
        case_id_col=case_index,
        time_col=time_col,
        activity_col=core_event,
        length_buckets=length_buckets
    )

    stats["dataset"] = dataname
    stats_rows.append(stats)

Processing helpdesk
Processing sepsis
Processing BPI13I
Processing BPI20
Processing BPI13C
Processing BPI17


In [75]:
import math
def in_bucket(length, min_len, max_len):
    if max_len is None:
        return length >= min_len
    return (length >= min_len) and (length < max_len)
    
def get_bucket_size(train_pairs, min_len, max_len):
    return sum(
        1
        for p in train_pairs
        if in_bucket(p["metadata"]["case_length"], min_len, max_len)
    )


def compute_bucket_k(bucket_size, min_k=3, max_k=10, scale=0.4):
    if bucket_size <= 0:
        return 0
    k = min(max_k, max(min_k, math.floor(0.2 * bucket_size)))
    return k

In [79]:
df = pd.read_csv(f"../output/data_processed/{dataname}_full.csv")
df = add_length_bucket(
            df,
            length_buckets=length_buckets,
            case_id_col=case_index,
            time_col=time_col,
            bucket_col="length_bucket",
            pos_col="pos"
        )

In [37]:
stats_df = pd.DataFrame(stats_rows)

# reorder columns (optional but cleaner)
cols = ["dataset", "cases", "events", "activities",
        "avg_len", "min_len", "max_len",
        "avg_duration", "avg_inter_event"] + list(length_buckets.keys())

stats_df = stats_df[[c for c in cols if c in stats_df.columns]]

In [39]:
stats_df.to_csv("../output/metrics/dataset_statistics.csv", index=False)